<a id="1"></a>
# <p style="padding:10px;background-color:#A23434;margin:0;color:#CDCA34;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Importing Libraries</p>

In [1]:
# %load_ext cudf.pandas
import dias.rewriter

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
sns.set()

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

<a id="1"></a>
# <p style="padding:10px;background-color:#A23434;margin:0;color:#CDCA34;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Reading Dataset</p>

In [3]:
start = time.time()

In [4]:
df = pd.read_csv("../input/adidas-us-retail-products-dataset/adidas.csv")

# -- STEFANOS -- Replicate Data

In [5]:
factor = 1000
df = pd.concat([df]*factor)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 845000 entries, 0 to 844
Data columns (total 20 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   url             845000 non-null  object 
 1   name            845000 non-null  object 
 2   sku             845000 non-null  object 
 3   selling_price   845000 non-null  int64  
 4   original_price  829000 non-null  object 
 5   currency        845000 non-null  object 
 6   availability    845000 non-null  object 
 7   color           845000 non-null  object 
 8   category        845000 non-null  object 
 9   source          845000 non-null  object 
 10  source_website  845000 non-null  object 
 11  breadcrumbs     845000 non-null  object 
 12  description     845000 non-null  object 
 13  brand           845000 non-null  object 
 14  images          845000 non-null  object 
 15  country         845000 non-null  object 
 16  language        845000 non-null  object 
 17  average_rating  84

<a id="1"></a>
# <p style="padding:10px;background-color:#A23434;margin:0;color:#CDCA34;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Basic Information</p>

In [6]:
# First 5 records in the DataFrame

df.head()

,url,name,sku,selling_price,original_price,currency,availability,color,category,source,source_website,breadcrumbs,description,brand,images,country,language,average_rating,reviews_count,crawled_at
0,https://www.adidas.com/us/beach-shorts/FJ5089....,Beach Shorts,FJ5089,40,NaN,USD,InStock,Black,Clothing,adidas United States,https://www.adidas.com,Women/Clothing,Splashing in the surf. Making memories with yo...,adidas,"https://assets.adidas.com/images/w_600,f_auto,...",USA,en,4.5,35,2021-10-23 17:50:17.331255
1,https://www.adidas.com/us/five-ten-kestrel-lac...,Five Ten Kestrel Lace Mountain Bike Shoes,BC0770,150,NaN,USD,InStock,Grey,Shoes,adidas United States,https://www.adidas.com,Women/Shoes,Lace up and get after it. The Five Ten Kestrel...,adidas,"https://assets.adidas.com/images/w_600,f_auto,...",USA,en,4.8,4,2021-10-23 17:50:17.423830
2,https://www.adidas.com/us/mexico-away-jersey/G...,Mexico Away Jersey,GC7946,70,NaN,USD,InStock,White,Clothing,adidas United States,https://www.adidas.com,Kids/Clothing,"Clean and crisp, this adidas Mexico Away Jerse...",adidas,"https://assets.adidas.com/images/w_600,f_auto,...",USA,en,4.9,42,2021-10-23 17:50:17.530834
3,https://www.adidas.com/us/five-ten-hiangle-pro...,Five Ten Hiangle Pro Competition Climbing Shoes,FV4744,160,NaN,USD,InStock,Black,Shoes,adidas United States,https://www.adidas.com,Five Ten/Shoes,The Hiangle Pro takes on the classic shape of ...,adidas,"https://assets.adidas.com/images/w_600,f_auto,...",USA,en,3.7,7,2021-10-23 17:50:17.615054
4,https://www.adidas.com/us/mesh-broken-stripe-p...,Mesh Broken-Stripe Polo Shirt,GM0239,65,NaN,USD,InStock,Blue,Clothing,adidas United States,https://www.adidas.com,Men/Clothing,Step up to the tee relaxed. This adidas golf p...,adidas,"https://assets.adidas.com/images/w_600,f_auto,...",USA,en,4.7,11,2021-10-23 17:50:17.702680


In [7]:
# Checking for Null Values in the DataFrame

df.isna().sum()

# There 16 Null Values for Original_price Column

url                   0
name                  0
sku                   0
selling_price         0
original_price    16000
currency              0
availability          0
color                 0
category              0
source                0
source_website        0
breadcrumbs           0
description           0
brand                 0
images                0
country               0
language              0
average_rating        0
reviews_count         0
crawled_at            0
dtype: int64

In [8]:
# Shape of the DataFrame

df.shape

(845000, 20)

In [9]:
# Information about the DataFrame

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 845000 entries, 0 to 844
Data columns (total 20 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   url             845000 non-null  object 
 1   name            845000 non-null  object 
 2   sku             845000 non-null  object 
 3   selling_price   845000 non-null  int64  
 4   original_price  829000 non-null  object 
 5   currency        845000 non-null  object 
 6   availability    845000 non-null  object 
 7   color           845000 non-null  object 
 8   category        845000 non-null  object 
 9   source          845000 non-null  object 
 10  source_website  845000 non-null  object 
 11  breadcrumbs     845000 non-null  object 
 12  description     845000 non-null  object 
 13  brand           845000 non-null  object 
 14  images          845000 non-null  object 
 15  country         845000 non-null  object 
 16  language        845000 non-null  object 
 17  average_rating  84

<a id="1"></a>
# <p style="padding:10px;background-color:#A23434;margin:0;color:#CDCA34;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Statistical Information</p>

In [10]:
# Correlation Between Columns

df.corr(numeric_only=True)

,selling_price,average_rating,reviews_count
selling_price,1.000000,-0.213004,0.102654
average_rating,-0.213004,1.000000,0.023585
reviews_count,0.102654,0.023585,1.000000


In [114]:
## DIAS_VERBOSE
# Statistical informtion about the DataFrame

df.describe().T

,count,mean,std,min,25%,50%,75%,max
selling_price,829000.0,52.911942,31.132766,9.00,28.0,48.0,68.00,240.0
original_price,829000.0,69.008444,40.465723,14.00,35.0,65.0,90.00,300.0
average_rating,829000.0,4.608323,0.292513,1.00,4.5,4.7,4.80,5.0
reviews_count,829000.0,433.943305,1238.942824,1.00,20.0,72.0,319.00,11750.0
Discount,829000.0,16.096502,11.763575,2.00,8.0,13.0,20.00,84.0
Discount(%),829000.0,22.727479,6.529758,7.14,20.0,20.0,29.23,50.0


### Dias did not rewrite code

<a id="1"></a>
# <p style="padding:10px;background-color:#A23434;margin:0;color:#CDCA34;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Feature Engineering</p>

In [115]:
# DIAS_VERBOSE
# The Percentage of Missing Values in the 'original_price' Column

100 * (df['original_price'].isna().sum() / len(df))

# As the percntage of Null Records is less than 5%, hence dropping the Null record Rows

0.0

### Dias did not rewrite code

In [13]:
# Dropping Null Values in the DataFrame

df.dropna(inplace=True, axis=0)

In [14]:
# Checking for Null Values in the DataFrame

df.isna().sum()

url               0
name              0
sku               0
selling_price     0
original_price    0
currency          0
availability      0
color             0
category          0
source            0
source_website    0
breadcrumbs       0
description       0
brand             0
images            0
country           0
language          0
average_rating    0
reviews_count     0
crawled_at        0
dtype: int64

In [15]:
# Dropping 'currency' column as all records have 'USD' as currency
# Dropping 'source' column as all records have 'adidas United States' as value
# Dropping 'brand', 'country', 'language' columns as all records have same value

df.drop([ 'brand', 'country', 'language', 'source_website', 'images', 'crawled_at', 'url', 'sku', 'currency','source', 'description'], axis=1, inplace=True)

In [16]:
# First 5 records in the DataFrame

df.head()

,name,selling_price,original_price,availability,color,category,breadcrumbs,average_rating,reviews_count
15,Essentials Loose Logo Tank Top,20,$25,InStock,Purple,Clothing,Women/Clothing,4.8,116
16,Essentials Loose Logo Tank Top,20,$25,InStock,Pink,Clothing,Women/Clothing,4.8,116
18,Essentials Loose Logo Tank Top,20,$25,InStock,Green,Clothing,Women/Clothing,4.8,116
19,Formotion Sculpt Tights,48,$80,InStock,Blue,Clothing,Women/Clothing,4.2,144
20,Marvel X Ghosted.3 Firm Ground Cleats,64,$80,InStock,Blue,Shoes,Soccer/Shoes,4.4,160


In [17]:
# Removing '$' from the DataFrame

df['original_price'] = df['original_price'].str.split('$')
df['original_price'] = df['original_price'].str[1]

In [18]:
# First 5 records in the DataFrame

df.head()

,name,selling_price,original_price,availability,color,category,breadcrumbs,average_rating,reviews_count
15,Essentials Loose Logo Tank Top,20,25,InStock,Purple,Clothing,Women/Clothing,4.8,116
16,Essentials Loose Logo Tank Top,20,25,InStock,Pink,Clothing,Women/Clothing,4.8,116
18,Essentials Loose Logo Tank Top,20,25,InStock,Green,Clothing,Women/Clothing,4.8,116
19,Formotion Sculpt Tights,48,80,InStock,Blue,Clothing,Women/Clothing,4.2,144
20,Marvel X Ghosted.3 Firm Ground Cleats,64,80,InStock,Blue,Shoes,Soccer/Shoes,4.4,160


In [19]:
# Shape of the DataFrame

df.shape

(829000, 9)

In [20]:
# Checking the Data Types for the columns

df.dtypes

name               object
selling_price       int64
original_price     object
availability       object
color              object
category           object
breadcrumbs        object
average_rating    float64
reviews_count       int64
dtype: object

In [113]:
# DIAS_VERBOSE
# Creating 'Category' Column

df['Category'] = df['breadcrumbs'].str.split("/")
df['Category'] = df['Category'].str[0]

KeyError: 'breadcrumbs'


### Dias rewrote code:
<br />

```python
if type(df['breadcrumbs']) != pd.Series:
    _REWR_res = df['breadcrumbs'].str.split('/')
else:
    _REWR_targ_0 = []
    _REWR_ls = df['breadcrumbs'].tolist()
    for _REWR_s in _REWR_ls:
        _REWR_spl = _REWR_s.split('/')
        _REWR_targ_0.append(_REWR_spl[0])
    _REWR_res = _REWR_targ_0
df['Category'] = _REWR_res

```


In [22]:
# Creating 'Product Type' Column

df['Product_Type'] = df['breadcrumbs'].str.split("/")
df['Product_Type'] = df['Product_Type'].str[1]

In [23]:
# Droping 'breadcrumbs' Column

df.drop(['breadcrumbs', 'category'], axis=1, inplace=True)

In [24]:
# Changing the DataType of 'original_price' from object to int64

df['original_price'] = df['original_price'].astype('int64')

In [25]:
# Checking the Data Types for the columns

df.dtypes

name               object
selling_price       int64
original_price      int64
availability       object
color              object
average_rating    float64
reviews_count       int64
Category           object
Product_Type       object
dtype: object

<a id="1"></a>
# <p style="padding:10px;background-color:#A23434;margin:0;color:#CDCA34;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">EDA and Data Visualization</p>

# Heat-Map

In [26]:
# STEFANOS: Disable plotting
# sns.heatmap(df.corr(), annot=True, cmap='magma')
# plt.show();

# Distribution Plots

In [27]:
# Distribution Plot for selling price

# STEFANOS: Disable plotting
# sns.displot(df['selling_price'], kde=True);

# The Below Graph is 'Right Skewed' with Majority of Data falling between 10-70.

In [28]:
# Distribution Plot for origial price

# STEFANOS: Disable plotting
# sns.displot(df['original_price'], kde=True);

# The Below Graph is 'Right Skewed' with Majority of Data falling between 10-90.

In [29]:
# Distribution Plot for average rating

# STEFANOS: Disable plotting
# sns.displot(df['average_rating'], kde=True);

# The Below Graph is 'Left Skewed' with Majority of Data falling between 4.2-5.0.

In [30]:
# Distribution Plot for reviews count

# STEFANOS: Disable plotting
# sns.displot(df['reviews_count'], kde=True);

# The Below Graph is 'Right Skewed' with Majority of Data falling below 600.

# Product Name

In [31]:
# No.of Unique Products in the DataFrame

df['name'].nunique()

417

In [32]:
# Top 10 - Products sold in the DataFrame

df['name'].value_counts()[:10]

name
ZX 1K Boost Shoes          24000
ZX 2K Boost Shoes          18000
Superstar Shoes            15000
EQ21 Run Shoes             13000
Racer TR21 Shoes           12000
ZX 2K Boost 2.0 Shoes      10000
Supernova+ Shoes            9000
4D Fusio Shoes              9000
Adilette Comfort Slides     9000
Swift Run X Shoes           9000
Name: count, dtype: int64

In [33]:
# Top 5 - Products sold in the DataFrame


# STEFANOS: Disable plotting
# plt.figure(figsize=(12,4))
# df['name'].value_counts()[:5].plot(kind='barh', color={'#864879','#2D4263','#C84B31', '#ECDBBA', '#B3541E'})
# plt.ylabel("Product Name")
# plt.xlabel("No.of Units Sold")
# plt.title("Product Name Vs. No.of Units Sold")
# plt.show();
df['name'].value_counts()[:5]

name
ZX 1K Boost Shoes    24000
ZX 2K Boost Shoes    18000
Superstar Shoes      15000
EQ21 Run Shoes       13000
Racer TR21 Shoes     12000
Name: count, dtype: int64

#### The Product '**ZX 1K Boost Shoes**' is the most sold Product/Recorded in the DataFrame with '**24**' units.

# Selling Price

In [34]:
# No.of Unique selling price in the DataFrame

df['selling_price'].nunique()

68

In [35]:
# Top 15 - Products interms of similar selling price in the DataFrame

df['selling_price'].value_counts()[:15]

selling_price
56     54000
20     50000
52     42000
32     40000
36     39000
28     35000
72     30000
60     29000
24     29000
70     28000
44     26000
40     25000
64     22000
48     22000
105    21000
Name: count, dtype: int64

In [36]:
# Top 5 - Products interms of similar selling price in the DataFrame

# STEFANOS: Disable plotting
# plt.figure(figsize=(12,4))
# df['selling_price'].value_counts()[:5].plot(kind='barh', color={'#864879','#2D4263','#C84B31', '#ECDBBA', '#B3541E'})
# plt.ylabel("Product Selling Price ($USD)")
# plt.xlabel("No.of Products")
# plt.title("Product Selling Price ($USD) Vs. No.of Products")
# plt.show();
df['selling_price'].value_counts()[:5]

selling_price
56    54000
20    50000
52    42000
32    40000
36    39000
Name: count, dtype: int64

In [37]:
# Top 15 - Products interms of similar selling price in the DataFrame

# STEFANOS: Disable plotting
# labels = df['selling_price'].value_counts().head(15).index
# pie, ax = plt.subplots(figsize=[20,8])
# plt.pie(x=df['selling_price'].value_counts().head(15).values, autopct="%.1f%%", explode=[0.06]*15, labels=labels, pctdistance=0.5)
# plt.title("Top 15 - Products interms of similar selling price ($USD)", fontsize=14);

## Maximum Selling Price

In [38]:
# Maximum Selling Price

print("The Maximun Selling Price is:",df['selling_price'].max(),"USD")

The Maximun Selling Price is: 240 USD


## Minimum Selling Price

In [39]:
# Minimum Selling Price

print("The Minimum Selling Price is:",df['selling_price'].min(),"USD")

The Minimum Selling Price is: 9 USD


## Average Selling Price

In [40]:
# Average Selling Price

print("The Average Selling Price is:",round(df['selling_price'].mean(),2),"USD")

The Average Selling Price is: 52.91 USD


In [41]:
# Top 10 - Highest Selling Prices in USD

set(df['selling_price'].sort_values(ascending=False)[:30].values)

{240}

In [42]:
# Top 10 - Least Selling Prices in USD

set(df['selling_price'].sort_values()[:40].values)

{9}

#### Highest Selling price is '**240 USD**' and Least is '**9 USD**' and product(s) with selling price of '**56 USD**' is sold '**54**' times.

# Original Price

In [43]:
# No.of Unique original price in the DataFrame

df['original_price'].nunique()

42

In [44]:
# Top 15 - Products interms of similar original price in the DataFrame

df['original_price'].value_counts()[:15]

original_price
65     68000
25     57000
100    55000
45     55000
30     49000
90     48000
70     47000
80     46000
35     40000
120    35000
50     33000
60     32000
40     32000
55     32000
150    29000
Name: count, dtype: int64

In [45]:
# Top 5 - Products interms of similar origial price in the DataFrame

# STEFANOS: Disable plotting
# plt.figure(figsize=(12,4))
# df['original_price'].value_counts()[:5].plot(kind='barh', color={'#864879','#2D4263','#C84B31', '#ECDBBA', '#B3541E'})
# plt.ylabel("Original Product Price ($USD)")
# plt.xlabel("No.of Products")
# plt.title("Original Product Price ($USD) Vs. No.of Products")
# plt.show();
df['original_price'].value_counts()[:5]

original_price
65     68000
25     57000
100    55000
45     55000
30     49000
Name: count, dtype: int64

In [46]:
# Top 15 - Products interms of similar Original price in the DataFrame

# STEFANOS: Disable plotting
# labels = df['original_price'].value_counts().head(15).index
# pie, ax = plt.subplots(figsize=[20,8])
# plt.pie(x=df['original_price'].value_counts().head(15).values, autopct="%.1f%%", explode=[0.06]*15, labels=labels, pctdistance=0.5)
# plt.title("Top 15 - Products interms of similar original price ($USD)", fontsize=14);

## Maximum Original Price

In [47]:
# Maximum Original Price

print("The Maximun Original Price is:",df['original_price'].max(),"USD")

The Maximun Original Price is: 300 USD


## Minimum Original Price

In [48]:
# Minimum Original Price

print("The Minimum Original Price is:",df['original_price'].min(),"USD")

The Minimum Original Price is: 14 USD


## Average Original Price

In [49]:
# Average Original Price

print("The Average Original Price is:",round(df['original_price'].mean(),2),"USD")

The Average Original Price is: 69.01 USD


In [50]:
# Top 10 - Highest Original Prices in USD

set(df['original_price'].sort_values(ascending=False)[:80].values)

{300}

In [51]:
# Top 10 - Least Original Prices in USD

set(df['original_price'].sort_values().values[:100])

{14}

#### Highest Original price is '**300 USD**' and Least is '**14 USD**' and product(s) with selling price of '**65 USD**' is sold '**68**' times.

# Discount

In [52]:
# Calculating Discount

df['Discount'] = df['original_price'] - df['selling_price']

In [53]:
# No.of Unique Discount in the DataFrame

df['Discount'].nunique()

41

In [54]:
# Top 15 - Highest Discount Amount in the DataFrame

list(set(df['Discount'].unique()))[-15::]

[32, 33, 36, 37, 39, 40, 42, 45, 48, 50, 54, 60, 72, 75, 84]

In [55]:
# Top 15 - Least Discount Amount in the DataFrame

list(set(df['Discount'].unique()))[:15]

[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

In [56]:
# Top 15 - Most Given Discount Amount in the DataFrame

df['Discount'].value_counts()[:15]

Discount
5     73000
9     61000
13    57000
10    40000
7     39000
14    38000
6     36000
18    33000
12    33000
8     32000
30    30000
15    29000
16    29000
4     28000
24    28000
Name: count, dtype: int64

In [57]:
# Top 5 - Most Given Discount Amount in the DataFrame

# STEFANOS: Disable plotting
# plt.figure(figsize=(12,4))
# df['Discount'].value_counts()[:5].plot(kind='barh', color={'#864879','#2D4263','#C84B31', '#ECDBBA', '#B3541E'})
# plt.ylabel("Discount Amount")
# plt.xlabel("No.of times Particular Disount Amount Given")
# plt.title("Discount Amount Vs. No.of times Particular Disount Amount Given")
# plt.show();
df['Discount'].value_counts()[:5]

Discount
5     73000
9     61000
13    57000
10    40000
7     39000
Name: count, dtype: int64

In [58]:
# Top 15 - Most Given Discount Amount in the DataFrame

labels = df['Discount'].value_counts().head(15).index
# STEFANOS: Disable plotting
# pie, ax = plt.subplots(figsize=[20,8])
# plt.pie(x=df['Discount'].value_counts().head(15).values, autopct="%.1f%%", explode=[0.06]*15, labels=labels, pctdistance=0.5)
# plt.title("Top 15 - Most Given Discount Amount in the DataFrame", fontsize=14);
_ = df['Discount'].value_counts().head(15).values

#### Highest Discount Amount is '**84 USD**' and Least is '**2 USD**' and product(s) with Disount Amount of '**5 USD**' is sold '**73**' times.

# Discount Percentage(%)

In [116]:
# DIAS_VERBOSE
# Calculating Discount percentage(%)

df['Discount(%)'] = round(((df['original_price'] - df['selling_price']) / (df['original_price']))*100,2)

### Dias did not rewrite code

In [60]:
# No.of Unique Discount Percentage(%) in the DataFrame

df['Discount(%)'].nunique()

40

In [61]:
# Top 15 - Highest Discount Percentage(%) in the DataFrame

top_discount_percnet = list(set(df['Discount(%)'].unique()))
top_discount_percnet.sort(reverse=True)
print(top_discount_percnet[:15])

[50.0, 49.33, 49.23, 40.0, 39.53, 35.71, 30.0, 29.41, 29.33, 29.31, 29.23, 29.17, 29.09, 28.89, 28.85]


In [62]:
# Top 15 - Least Discount Percentage(%) in the DataFrame

least_discount_percnet = list(set(df['Discount(%)'].unique()))
least_discount_percnet.sort(reverse=False)
print(least_discount_percnet[:15])

[7.14, 8.0, 8.57, 8.89, 9.09, 9.23, 9.33, 10.0, 14.29, 16.67, 17.39, 17.86, 18.18, 18.6, 18.75]


In [63]:
# Top 15 - Most Given Discount Percentage(%) in the DataFrame

df['Discount(%)'].value_counts().head(15)

Discount(%)
20.00    454000
30.00    173000
10.00     44000
29.23     20000
28.57     15000
28.89     15000
40.00     11000
17.86     11000
27.78      8000
29.41      8000
28.00      7000
18.18      6000
29.33      5000
8.89       5000
7.14       4000
Name: count, dtype: int64

In [64]:
# Top 5 - Most Given Discount Percentage in the DataFrame

# STEFANOS: Disable plotting
# plt.figure(figsize=(12,4))
# df['Discount(%)'].value_counts().head(5).plot(kind='barh', color={'#864879','#2D4263','#C84B31', '#ECDBBA', '#B3541E'})
# plt.ylabel("Discount Percentage")
# plt.xlabel("No.of times Particular Disount Percentage Given")
# plt.title("Discount Percentage Vs. No.of times Particular Disount Percentage Given")
# plt.show();
df['Discount(%)'].value_counts().head(5)

Discount(%)
20.00    454000
30.00    173000
10.00     44000
29.23     20000
28.57     15000
Name: count, dtype: int64

In [65]:
# Top 15 - Most Given Discount Percentage in the DataFrame

labels = df['Discount(%)'].value_counts().head(15).index
# STEFANOS: Disable plotting
# pie, ax = plt.subplots(figsize=[20,15])
# plt.pie(x=df['Discount(%)'].value_counts().head(15).values, autopct="%.1f%%", explode=[0.06]*15, labels=labels, pctdistance=0.5)
# plt.title("Top 15 - Most Given Discount Percenatage in the DataFrame", fontsize=14);
# plt.tight_layout();
_=df['Discount(%)'].value_counts().head(15).values

#### Highest Discount% is '**50%**' and Least is '**7.14%**' and product(s) with Disount Percentage of '**20**' is sold '**254**' times.

# Color

In [66]:
# Unique colors in the DataFrame

df['color'].unique()

array(['Purple', 'Pink', 'Green', 'Blue', 'Grey', 'White', 'Yellow',
       'Black', 'Red', 'Multicolor', 'Gold', 'Burgundy', 'Beige',
       'Orange', 'Multi', 'Turquoise', 'Silver', 'Brown'], dtype=object)

In [67]:
# No.of Unique colors in the DataFrame

df['color'].nunique()

18

In [68]:
# No.of Products for each Color

df['color'].value_counts()

color
White         218000
Black         179000
Blue          102000
Grey           79000
Pink           62000
Green          59000
Purple         31000
Red            25000
Multicolor     20000
Yellow         17000
Orange         11000
Burgundy        9000
Beige           6000
Multi           4000
Gold            3000
Turquoise       2000
Silver          1000
Brown           1000
Name: count, dtype: int64

In [69]:
# No.of Products for each Color

labels = df['color'].value_counts().index
# STEFANOS: Disable plotting
# pie, ax = plt.subplots(figsize=[20,15])
# plt.pie(x=df['color'].value_counts().values, autopct="%.1f%%", explode=[0.03]*18, labels=labels, pctdistance=0.5)
# plt.title("No.of Products for each Color in the DataFrame", fontsize=14);
# plt.tight_layout();
_ = x=df['color'].value_counts().values

# Which Color is most popular among genders?

In [70]:
# Popular Color in Kids Category

df.groupby(['Category','color']).size()['Kids']

color
Black         20000
Blue          12000
Green          4000
Grey           9000
Multicolor     1000
Orange         1000
Pink          13000
Purple         3000
Red            5000
White         26000
Yellow         3000
dtype: int64

In [117]:
# DIAS_VERBOSE
# Popular Color in Womens Category

df.groupby(['Category','color']).size()['Women']

color
Beige          1000
Black         67000
Blue          28000
Burgundy       5000
Gold           1000
Green         26000
Grey          23000
Multi          3000
Multicolor    10000
Orange         2000
Pink          44000
Purple        26000
Red           10000
Turquoise      1000
White         92000
Yellow         3000
dtype: int64

### Dias did not rewrite code

In [72]:
# Popular Color in Mens Category

df.groupby(['Category','color']).size()['Men']

color
Beige          5000
Black         61000
Blue          47000
Burgundy       4000
Gold           2000
Green         22000
Grey          39000
Multi          1000
Multicolor     7000
Orange         7000
Pink           1000
Purple         2000
Red            8000
White         67000
Yellow         6000
dtype: int64

#### The top color in **Mens** category is '**White**' with '67' sales,the top color in **Womens** category is '**White**' with '92' sales and the top color in **Kids** category is '**White**' with '26' sales

# Availability

In [73]:
# Unique Availability values in the DataFrame

df['availability'].unique()

array(['InStock', 'OutOfStock'], dtype=object)

In [74]:
# No.of Unique Availability values in the DataFrame

df['availability'].nunique()

2

In [75]:
# No.of Products according to Avaialability

df['availability'].value_counts()

availability
InStock       826000
OutOfStock      3000
Name: count, dtype: int64

#### There are '**826**' products which are '**InStock**' and '**3**' are '**OutOfStock**'.

## Product Type

In [76]:
# Unique Product Type values in the DataFrame

df['Product_Type'].unique()

array(['Clothing', 'Shoes', 'Accessories'], dtype=object)

In [77]:
# No.of Unique Product Type values in the DataFrame

df['Product_Type'].nunique()

3

In [78]:
# No.of Products according to Product Type

df['Product_Type'].value_counts()

Product_Type
Shoes          422000
Clothing       328000
Accessories     79000
Name: count, dtype: int64

In [79]:
# No.of Products according to Product Type

# STEFANOS: Disable plotting
# plt.figure(figsize=(12,4))
# df['Product_Type'].value_counts().plot(kind='barh', color={'#864879','#2D4263','#C84B31'})
# plt.ylabel("Product Type")
# plt.xlabel("No.of Products Sold According to Particular - Product Type")
# plt.title("Product Type Vs. No.of Products Sold According to Particular - Product Type")
# plt.show();
df['Product_Type'].value_counts()

Product_Type
Shoes          422000
Clothing       328000
Accessories     79000
Name: count, dtype: int64

In [118]:
# DIAS_VERBOSE
# No.of Products according to Product Type

labels = df['Product_Type'].value_counts().index
# STEFANOS: Disable plotting
# pie, ax = plt.subplots(figsize=[10,4])
# plt.pie(x=df['Product_Type'].value_counts().values, autopct="%.1f%%", explode=[0.05]*3, labels=labels, pctdistance=0.5)
# plt.title("No.of Products according to Product Type in the DataFrame", fontsize=14);
# plt.tight_layout();
_ = df['Product_Type'].value_counts().values

### Dias did not rewrite code

## Maximum Selling Price for Each Product Type

In [81]:
df.groupby(['Product_Type', 'Category']).max()['selling_price']

Product_Type  Category  
Accessories   Kids           13
              Men            23
              Originals     196
              Running        21
              Soccer         32
              Training       36
              Women          40
Clothing      Essentials     44
              Kids           60
              Men           240
              Originals      60
              Sportswear     72
              Women         128
Shoes         Essentials     60
              Kids          128
              Men           180
              Originals     180
              Running       144
              Soccer        200
              Swim           48
              Women         126
Name: selling_price, dtype: int64

## Minimum Selling Price for Each Product Type

In [112]:

df.groupby(['Product_Type', 'Category']).min()['selling_price']

Product_Type  Category  
Accessories   Kids          13
              Men           10
              Originals      9
              Running       21
              Soccer        18
              Training      12
              Women         10
Clothing      Essentials    44
              Kids          19
              Men           18
              Originals     60
              Sportswear    72
              Women         16
Shoes         Essentials    60
              Kids          24
              Men           20
              Originals     44
              Running       98
              Soccer        64
              Swim          20
              Women         20
Name: selling_price, dtype: int64

### Dias did not rewrite code

## Average Selling Price for Each Product Type

In [83]:
df.groupby(['Product_Type', 'Category']).mean(numeric_only=True)['selling_price']

Product_Type  Category  
Accessories   Kids           13.000000
              Men            16.000000
              Originals      43.727273
              Running        21.000000
              Soccer         25.000000
              Training       21.360000
              Women          19.500000
Clothing      Essentials     44.000000
              Kids           30.555556
              Men            39.760000
              Originals      60.000000
              Sportswear     72.000000
              Women          38.924855
Shoes         Essentials     60.000000
              Kids           50.956522
              Men            79.676056
              Originals      89.400000
              Running       119.200000
              Soccer         84.000000
              Swim           32.142857
              Women          61.405229
Name: selling_price, dtype: float64

#### The Product Type with most sales is '**Shoes**', and Maximum Selling Price of '**240$**' according to Selling Price. 

# Category

In [111]:

# Unique Category values in the DataFrame

df['Category'].unique()

array(['Women', 'Soccer', 'Men', 'Kids', 'Essentials', 'Originals',
       'Training', 'Swim', 'Running', 'Sportswear'], dtype=object)

### Dias did not rewrite code

In [85]:
# No.of Unique Category values in the DataFrame

df['Category'].nunique()

10

In [86]:
# No.of Products according to Category

df['Category'].value_counts()

Category
Women         342000
Men           279000
Kids           97000
Originals      58000
Training       25000
Soccer         12000
Swim            7000
Running         6000
Essentials      2000
Sportswear      1000
Name: count, dtype: int64

In [110]:

# Top-5 Products according to Category

# STEFANOS: Disable plotting
# plt.figure(figsize=(12,4))
# df['Category'].value_counts().head(5).plot(kind='barh', color={'#864879','#2D4263','#C84B31', '#ECDBBA', '#B3541E'})
# plt.ylabel("Category")
# plt.xlabel("No.of Products Sold According to Particular - Category")
# plt.title("Top 5 - Products Sold according to Category in the DataFrame")
# plt.show();
df['Category'].value_counts().head(5)

Category
Women        342000
Men          279000
Kids          97000
Originals     58000
Training      25000
Name: count, dtype: int64

### Dias did not rewrite code

In [88]:
# Top-5 Products according to Category

labels = df['Category'].value_counts().head(5).index
# STEFANOS: Disable plotting
# pie, ax = plt.subplots(figsize=[10,4])
# plt.pie(x=df['Category'].value_counts().head(5).values, autopct="%.1f%%", explode=[0.05]*5, labels=labels, pctdistance=0.5)
# plt.title("Top 5 - Products Sold according to Category in the DataFrame", fontsize=14);
# plt.tight_layout();
_ = df['Category'].value_counts().head(5).values

#### The 'Category' with highest no.of Products Sold is '**Women**' with '**342 Products**', followed by 'Men' and 'Women'.

# Average Rating

In [89]:
# Unique 'Average Rating'in the DataFrame

df['average_rating'].unique()

array([4.8, 4.2, 4.4, 4.5, 4.6, 5. , 4.7, 4.3, 4. , 4.9, 3.9, 3.7, 3. ,
       4.1, 1. , 3.8])

In [90]:
# No.of Unique 'Average Rating'in the DataFrame

df['average_rating'].nunique()

16

In [91]:
# No.of time particular Average Rating Provided

df['average_rating'].value_counts()

average_rating
4.8    161000
4.7    159000
4.6    120000
4.5     99000
5.0     70000
4.2     51000
4.4     49000
4.9     44000
4.3     29000
3.9     14000
4.1     14000
4.0     12000
3.7      4000
3.0      1000
1.0      1000
3.8      1000
Name: count, dtype: int64

In [92]:
# Top 5 - Average Rating Provided

# STEFANOS: Disable plotting
# plt.figure(figsize=(12,4))
# df['average_rating'].value_counts().head(5).plot(kind='barh', color={'#864879','#2D4263','#C84B31', '#ECDBBA', '#B3541E'})
# plt.ylabel("Average Rating")
# plt.xlabel("No.of times particular rating provided")
# plt.title("Top 5 - Average Rating Provided in the DataFrame")
# plt.show();
df['average_rating'].value_counts().head(5)

average_rating
4.8    161000
4.7    159000
4.6    120000
4.5     99000
5.0     70000
Name: count, dtype: int64

In [93]:
# Top 5 - Average Rating Provided

labels = df['average_rating'].value_counts().head(5).index
# STEFANOS: Disable plotting
# pie, ax = plt.subplots(figsize=[10,4])
# plt.pie(x=df['average_rating'].value_counts().head(5).values, autopct="%.1f%%", explode=[0.05]*5, labels=labels, pctdistance=0.5)
# plt.title("# Top 5 - Average Rating Provided in the DataFrame", fontsize=14);
# plt.tight_layout();
_ = df['average_rating'].value_counts().head(5).values

In [94]:
# Maxmimum Average Rating

df['average_rating'].max()

5.0

In [95]:
# Minimum Average Rating

df['average_rating'].min()

1.0

In [96]:
# Mean Average Rating

round(df['average_rating'].mean(),2)

4.61

In [97]:
# Maximum Average Rating across 'Product Type' and 'Category'

df.groupby(['Product_Type', 'Category']).max()['average_rating']

Product_Type  Category  
Accessories   Kids          4.9
              Men           5.0
              Originals     5.0
              Running       4.7
              Soccer        4.8
              Training      5.0
              Women         5.0
Clothing      Essentials    4.7
              Kids          5.0
              Men           5.0
              Originals     5.0
              Sportswear    4.8
              Women         5.0
Shoes         Essentials    4.2
              Kids          5.0
              Men           5.0
              Originals     4.8
              Running       4.8
              Soccer        4.7
              Swim          4.7
              Women         5.0
Name: average_rating, dtype: float64

In [98]:
# Minimum Average Rating across 'Product Type' and 'Category'

df.groupby(['Product_Type', 'Category']).min()['average_rating']

Product_Type  Category  
Accessories   Kids          4.9
              Men           4.7
              Originals     4.5
              Running       4.7
              Soccer        4.8
              Training      4.6
              Women         4.6
Clothing      Essentials    4.7
              Kids          4.0
              Men           1.0
              Originals     5.0
              Sportswear    4.8
              Women         3.7
Shoes         Essentials    4.2
              Kids          4.3
              Men           3.9
              Originals     3.9
              Running       4.6
              Soccer        4.3
              Swim          4.4
              Women         3.8
Name: average_rating, dtype: float64

#### The **Maximum** Average Rating among all products is '**5.0**' and **Minimum** Average Rating among all products is '**1.0**'

# Reviews Count

In [99]:
# No.of Unique 'Reviews Count' in the DataFrame

df['reviews_count'].nunique()

211

In [100]:
# Top 10 - 'Reviews Count' and no.of particular 'Reviews Count' occurances

df['reviews_count'].value_counts().head(10)

reviews_count
1      23000
2      21000
5      15000
9      15000
4      14000
8      14000
3      13000
562    12000
51     12000
454    12000
Name: count, dtype: int64

In [101]:
# Top 5 - 'Reviews Count' and no.of particular 'Reviews Count' occurances

# STEFANOS: Disable plotting
# plt.figure(figsize=(12,4))
# df['average_rating'].value_counts().head(5).plot(kind='barh', color={'#864879','#2D4263','#C84B31', '#ECDBBA', '#B3541E'})
# plt.ylabel('Reviews Count')
# plt.xlabel('no.of particular Reviews Count occurances')
# plt.title("Top 5 - 'Reviews Count' and no.of particular 'Reviews Count' occurances")
# plt.show();
df['average_rating'].value_counts().head(5)

average_rating
4.8    161000
4.7    159000
4.6    120000
4.5     99000
5.0     70000
Name: count, dtype: int64

In [102]:
# Maxmimum 'Reviews Count'

df['reviews_count'].max()

11750

In [103]:
# Minimum 'Reviews Count'

df['reviews_count'].min()

1

In [104]:
# Mean 'Reviews Count'

round(df['reviews_count'].mean(),2)

433.94

In [105]:
# Maximum 'Reviews Count' across 'Product Type' and 'Category'

df.groupby(['Product_Type', 'Category']).max()['reviews_count']

Product_Type  Category  
Accessories   Kids             41
              Men             120
              Originals       298
              Running          25
              Soccer           38
              Training        273
              Women           154
Clothing      Essentials       29
              Kids             74
              Men            1352
              Originals         2
              Sportswear       26
              Women           671
Shoes         Essentials      120
              Kids            602
              Men           11750
              Originals     11750
              Running         584
              Soccer          281
              Swim           7291
              Women          9636
Name: reviews_count, dtype: int64

In [106]:
# Minimum Reviews Count' across 'Product Type' and 'Category'

df.groupby(['Product_Type', 'Category']).min()['reviews_count']

Product_Type  Category  
Accessories   Kids           41
              Men             2
              Originals       1
              Running        25
              Soccer          5
              Training        1
              Women           2
Clothing      Essentials     29
              Kids            1
              Men             1
              Originals       2
              Sportswear     26
              Women           1
Shoes         Essentials    120
              Kids            3
              Men             1
              Originals      49
              Running       491
              Soccer         24
              Swim           79
              Women           3
Name: reviews_count, dtype: int64

#### The **Maximum** 'Reviews Count' among all products is '**11750**' and **Minimum** 'Reviews Count' among all products is '**1**'

# Reviews Count Vs. Average Rating

In [107]:
# STEFANOS: Disable plotting
# sns.scatterplot(data=df, x='average_rating', y='reviews_count')

In [108]:
end = time.time()
print(f"Total execution time {end-start}")

Total execution time 9.737637281417847


In [109]:
df.shape

(829000, 11)

#### The Correlation Between Reviews Count & Average Rating is '**0.024**' (positive correlation), from the graph we may can conclude that there are more number of reviews when particular product rated above **3.5** 

## End of the Notebook, Thanks for Watching, "**Do Upvote**" if it is helpful.